In [1]:
import sys

if sys.version_info[0] + 0.1 * sys.version_info[1] < 3.6:
    raise Exception("Must be using Python 3.6+ version")

In [2]:
%matplotlib inline
sys.path.append("..")

import wavelet_transform.utils.PCD_data as pcd
import wavelet_transform.utils.training_data as train
from wavelet_transform.utils.preprocessing import preprocess
import wavelet_transform.wpt.inverse_transform as inv
import wavelet_transform.utils.kmeans_results as results
from wavelet_transform.utils.machine_learning import fit_and_evaluate, train_k_means, assess_parameter

In [3]:
# Parameters for the wavelet transform
wavelet = "db20"
levels = None  # set to None to use max-level transform
mode = "symmetric"
use_min_entropy = True
min_level = 8

In [4]:
# k-means parameters
kmeans_params = {"init": "k-means++",
    "n_init":10,
    "max_iter":100,
    "tol":1e-8,
    "verbose":0,
    "algorithm":"lloyd",
    }

# training data
classifying_metrics = [
    "average energy",
    "kurtosis",
    # "crest factor",
    # "log average energy",
    # "level entropy",
    "spectral entropy",
    # "skewness"
]

# Set to None to select optimal number of clusters by silhouette score
no_clusters = 2
use_inverse_metrics = False

In [7]:
parameters = {
    "amplitude": 450,
    "frequency": 2,
    "duty cycle": 10,
    "prf": 100,  # pulse repetition frequency
    "experiment": "00004",
}

n_experiments = 5
frequency = 2e6 # parameters["frequency"] * 1e6
for experiment in (f"0000{i}" for i in range(n_experiments)):
    parameters["experiment"] = experiment
    dfbs = pcd.get_data(parameters)
    data, t_plot = dfbs["c2"]
    excitation_data, t_plot_ext = dfbs["c1"]

    result_id = (
        parameters["amplitude"], 
        frequency, 
        int(parameters["experiment"][-1])
    )

    sampling_rate = 1 / (t_plot[1] - t_plot[0])
    freq_threshold = 20e6  # the nodes with norm_freq_range above this threshold will not be kept
    freq_threshold /= (sampling_rate / 2)

    data_processed = preprocess(data, window=False)

    wpt_df, tr_obj = train.generate_training_data(
        data_processed, levels, wavelet, mode, 
        include_tr_obj=True, min_entropy=use_min_entropy,
        min_level=min_level, freq_threshold=freq_threshold
    )

    # Train k-means
    if use_inverse_metrics is True:
        inv_tr_obj = inv.InverseTransform(tr_obj)  # Use info from inverse transform object instead
        wpt_df_train = inv_tr_obj.info(leaf_nodes=True)
    else:
        wpt_df_train = wpt_df.copy()

    data_train, labels, data_train_normalised, centers, _, kmeans = train_k_means(
        wpt_df_train, classifying_metrics, no_clusters=no_clusters, kmeans_params=kmeans_params
    )

    # Save data
    additional_params = {
        "raw_data": data
    }

    results.save_data(
        result_id, data_train, tr_obj, centers, labels, 
        sampling_rate, freq_threshold, **additional_params
    )